<a href="https://colab.research.google.com/github/JulianSantos-LATAMAI/card-krueger-replication/blob/main/notebooks/02_Replication_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Julián Santos

ECON 5200 Midterm project

Phase:2 Replication Analysis


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
FILEPATH = '/content/public.dat'
df = pd.read_csv(FILEPATH)
print(df.head())

In [ ]:
#Load Clean Data

# Cell 1 - Imports
import pandas as pd
import numpy as np

# Cell 2 - Load Data
col_specs = [
    (0, 3),(3, 5),(5, 7),(7, 9),(9, 11),(11, 13),(13, 15),(15, 17),(17, 19),
    (19, 21),(21, 24),(24, 30),(30, 36),(36, 42),(42, 48),(48, 54),(54, 58),
    (58, 62),(62, 67),(67, 69),(69, 73),(73, 82),(82, 88),(88, 94),(94, 100),
    (100, 103),(103, 107),(107, 109),(109, 111),(111, 117),(117, 120),(120, 126),
    (126, 132),(132, 138),(138, 140),(140, 145),(145, 151),(151, 154),(154, 160),
    (160, 166),(166, 172),(172, 178),(178, 181),(181, 185),(185, 191),(191, 195)
]

col_names = [
    'SHEET','CHAIN','CO_OWNED','STATE','SOUTHJ','CENTRALJ','NORTHJ',
    'PA1','PA2','SHORE','NCALLS','EMPFT','EMPPT','NMGRS','WAGE_ST',
    'INCTIME','FIRSTINC','BONUS','PCTAFF','MEALS','OPEN','HRSOPEN',
    'PSODA','PFRY','PENTREE','NREGS','NREGS11','EMPFT2','EMPPT2',
    'DATE','NMGRS2','WAGE_ST2','INCTIME2','FIRSTIN2','BONUS2',
    'PCTAFF2','MEALS2','OPEN2R','HRSOPEN2','PSODA2','PFRY2',
    'PENTREE2','NREGS2','NREGS112','PSODA3','NREGS3'
]

df = pd.read_fwf('/content/public.dat', colspecs=col_specs, names=col_names, na_values='.')
df.head()

,SHEET,CHAIN,CO_OWNED,STATE,SOUTHJ,CENTRALJ,NORTHJ,PA1,PA2,SHORE,...,MEALS2,OPEN2R,HRSOPEN2,PSODA2,PFRY2,PENTREE2,NREGS2,NREGS112,PSODA3,NREGS3
0,46,1,0,0,0,0,0,1,0,0,...,26.0,0.0,08 1 2,6.5,16.5,1.03,NaN,NaN,0.94,4.0
1,49,2,0,0,0,0,0,1,0,0,...,13.0,0.0,05 0 2,10.0,13.0,1.01,0.0,0.89,2.35,4.0
2,506,2,1,0,0,0,0,1,0,0,...,19.0,0.0,25 . 1,11.0,11.0,0.95,0.0,0.74,2.33,4.0
3,56,4,1,0,0,0,0,1,0,0,...,26.0,0.0,15 0 2,10.0,12.0,0.92,0.0,0.79,0.87,2.0
4,61,4,1,0,0,0,0,1,0,0,...,13.0,0.0,15 0 2,10.0,12.0,1.01,0.0,0.84,0.95,2.0


In [ ]:
#Descri[ptive Table Reconstruction

# Building FTE variables
#FTE (Full Time Equivalent) = full time employees + 0.5 * part time employees + managers.

df['FTE']  = df['EMPFT']  + 0.5 * df['EMPPT']  + df['NMGRS']
df['FTE2'] = df['EMPFT2'] + 0.5 * df['EMPPT2'] + df['NMGRS2']
df['TREAT'] = df['STATE']

# Table Construction: Finding STDV and Mean
variables_w1 = ['FTE', 'WAGE_ST', 'PSODA', 'PFRY', 'PENTREE']
variables_w2 = ['FTE2', 'WAGE_ST2', 'PSODA2', 'PFRY2', 'PENTREE2']

means_w1 = df.groupby('STATE')[variables_w1].mean().round(2)
stds_w1  = df.groupby('STATE')[variables_w1].std().round(2)
means_w2 = df.groupby('STATE')[variables_w2].mean().round(2)
stds_w2  = df.groupby('STATE')[variables_w2].std().round(2)

table2 = pd.DataFrame({
    ('Wave 1 (Before)', 'NJ Mean'): means_w1.loc[1].values,
    ('Wave 1 (Before)', 'NJ Std'):  stds_w1.loc[1].values,
    ('Wave 1 (Before)', 'PA Mean'): means_w1.loc[0].values,
    ('Wave 1 (Before)', 'PA Std'):  stds_w1.loc[0].values,
    ('Wave 2 (After)',  'NJ Mean'): means_w2.loc[1].values,
    ('Wave 2 (After)',  'NJ Std'):  stds_w2.loc[1].values,
    ('Wave 2 (After)',  'PA Mean'): means_w2.loc[0].values,
    ('Wave 2 (After)',  'PA Std'):  stds_w2.loc[0].values,
}, index=['FTE', 'WAGE_ST', 'PSODA', 'PFRY', 'PENTREE'])

print(table2.to_string())

        Wave 1 (Before)                       Wave 2 (After)                      
                NJ Mean NJ Std PA Mean PA Std        NJ Mean NJ Std PA Mean PA Std
FTE               20.44   9.11   23.33  11.86           4.16   2.19    3.44   1.34
WAGE_ST            4.61   0.35    4.63   0.35           8.45   7.84    7.56   8.49
PSODA              1.06   0.08    0.97   0.07           8.17   2.16    7.88   2.16
PFRY               0.94   0.10    0.84   0.09          14.42   2.72   14.65   2.89
PENTREE            1.35   0.65    1.22   0.62           1.06   0.09    0.98   0.08


In [ ]:
#Simple Difference-In-Difference

# Mean FTE by state and before/after minimum wage hike
nj_before = df[df['TREAT']==1]['FTE'].mean()
nj_after  = df[df['TREAT']==1]['FTE2'].mean()
pa_before = df[df['TREAT']==0]['FTE'].mean()
pa_after  = df[df['TREAT']==0]['FTE2'].mean()

# Differences
nj_diff = nj_after - nj_before
pa_diff = pa_after - pa_before
did     = nj_diff - pa_diff

print(f"NJ Before: {nj_before:.2f}")
print(f"NJ After:  {nj_after:.2f}")
print(f"PA Before: {pa_before:.2f}")
print(f"PA After:  {pa_after:.2f}")
print(f"\nNJ Change:        {nj_diff:.2f}")
print(f"PA Change:        {pa_diff:.2f}")
print(f"\nDID Estimate:     {did:.2f}")

NJ Before: 20.44
NJ After:  4.16
PA Before: 23.33
PA After:  3.44

NJ Change:        -16.28
PA Change:        -19.89

DID Estimate:     3.61


In [ ]:
import statsmodels.formula.api as smf
import statsmodels.formula.api as smf

# Reshape to long format
wave1 = df[['SHEET','TREAT','FTE']].rename(columns={'FTE':'EMP'})
wave1['POST'] = 0

wave2 = df[['SHEET','TREAT','FTE2']].rename(columns={'FTE2':'EMP'})
wave2['POST'] = 1

df_long = pd.concat([wave1, wave2]).dropna()

# DID Regression
model = smf.ols('EMP ~ TREAT * POST', data=df_long)
results = model.fit(cov_type='cluster', cov_kwds={'groups': df_long['SHEET']})
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:                    EMP   R-squared:                       0.470
Model:                            OLS   Adj. R-squared:                  0.467
Method:                 Least Squares   F-statistic:                     388.8
Date:                Tue, 10 Mar 2026   Prob (F-statistic):          3.68e-118
Time:                        15:02:06   Log-Likelihood:                -1971.6
No. Observations:                 559   AIC:                             3951.
Df Residuals:                     555   BIC:                             3969.
Df Model:                           3                                         
Covariance Type:              cluster                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     23.3312      1.348     17.312      0.0


Standard OLS standard errors assume all observations are independent.
However, the two waves from the same store are correlated with each other.
We cluster standard errors at the store level (SHEET) to account for this,
which gives us more honest/conservative standard errors.

In [ ]:
# Extract and display the key DID coefficient
did_coef = results.params['TREAT:POST']
did_pval = results.pvalues['TREAT:POST']
did_ci   = results.conf_int().loc['TREAT:POST']

print(f"DID Estimate:   {did_coef:.3f}")
print(f"P-value:        {did_pval:.3f}")
print(f"95% CI:         [{did_ci[0]:.3f}, {did_ci[1]:.3f}]")

DID Estimate:   3.613
P-value:        0.014
95% CI:         [0.729, 6.497]


With a DID estimate of 3.613, this means that NJ minimum wage increase led to an increase of 3.6 FTE employees per store realtive to PA. With a P-Value of 0.014, that means that this results are statistically sginificant as well.